In [1]:
import duckdb
import pandas as pd
import sys
import spacy
import os
sys.path.append('..')

from src.utils import make_corpus, preprocess_spacy
from src.semantic import build_semantic_index, semantic_search
from src.bm25 import bm25_search, bm25_tokenize

# Build corpus

In [2]:
# Read data and drop missing values
c2 = duckdb.connect()
data = c2.execute(f"SELECT * FROM read_parquet('../data/raw/merged.parquet')").df()
data.dropna(subset=['product_title'], inplace=True)

In [3]:
# Extract fields for retrieval
cols = ['product_title', 'main_category', 'store', 'title', 'text']

corpus = make_corpus(df=data, cols=cols, asin="asin")

# Save indices

In [4]:
# preprocess corpus and save it
os.makedirs('data/processed', exist_ok=True)

# if corpus is already processed and saved, pass to save time
corpus_path = "../data/processed/corpus.csv"
if os.path.exists(corpus_path):
    corpus = pd.read_csv(corpus_path)
else:
    nlp = spacy.load("en_core_web_md", disable=["parser", "ner"])
    corpus["text"] = [preprocess_spacy(text) for text in nlp.pipe(corpus["text"])]
    corpus.to_csv(corpus_path)

In [5]:
# BM25 index
from rank_bm25 import BM25Okapi


# save the BM25 index into pickle file
import pickle

pickle_path = "../data/processed/bm25.pkl"

if not os.path.exists(pickle_path):
    # tokenize corpus
    products = corpus["text"]
    tokenized_products = [bm25_tokenize(p) for p in products]
    bm25 = BM25Okapi(tokenized_products)
    
    # save to pickle
    with open(pickle_path, "wb") as f:
        pickle.dump(bm25, f)

# load it
with open(pickle_path, "rb") as f:
    bm25 = pickle.load(f)

In [6]:
# Semantic index 
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
semantic_index_path = '../data/processed/embedding.faiss'

if not os.path.exists(semantic_index_path):
    build_semantic_index(corpus, model, semantic_index_path)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


# Retrieve results

In [7]:
queries = ["Wet wipes"]

In [8]:
for q in queries:
    print(f"QUERY: {q}\n")

    print("BM25 top results:")
    for rank, (product, score) in enumerate(bm25_search(q), bm25, data start=1):
        print(f"{rank}. ({score:.3f}) {product}")

    print("\nSemantic search top results:")
    display(semantic_search(q, semantic_index_path, model, data))

SyntaxError: invalid syntax. Perhaps you forgot a comma? (2390289734.py, line 5)